# Тема 10. Иерархическая МАС на LangGraph

**Модуль III · дополнительный пример**

В основных примерах модуля мультиагентная система собрана на «голом» OpenAI SDK. Здесь — **идиоматичная** реализация иерархической МАС на **LangGraph**: специалисты как готовые агенты, а **координатор — это граф**, где ветви-специалисты работают параллельно и сходятся в узел агрегации.

### Что показано
1. Специалисты с **ограниченными** наборами инструментов через `create_agent`.
2. **Граф-супервизор**: параллельный fan-out к специалистам + узел-координатор, агрегирующий результаты.
3. Аккумуляция результатов ветвей через reducer состояния.

### Базовые кирпичи (кратко; подробно — в примере M2/05)

Если вы не знакомы с LangChain/LangGraph, вот минимум, который используется в первых ячейках:
- **`ChatOpenAI(model, base_url, api_key, temperature)`** — объект модели (OpenAI-совместимый эндпоинт). `temperature=0` — детерминированные ответы.
- **`@tool`** — декоратор, превращающий Python-функцию в инструмент: имя берётся из имени функции, описание — из докстринга, схема аргументов — из аннотаций типов.
- **`create_agent(model, tools)`** — собирает готового ReAct-агента (внутри — граф LangGraph); у него есть `.invoke({"messages":[...]})`, а `result["messages"][-1].content` — финальный ответ.

Новое в этом примере — **своя схема состояния**, **reducer** и **параллельные ветви** графа; их разбираем ниже подробно.

### Доступ к модели
Параметры — из переменных окружения: `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

llm = ChatOpenAI(model=os.environ.get("LLM_MODEL", "openai/gpt-4.1-mini"),
                 base_url=os.environ.get("LLM_BASE_URL"),
                 api_key=os.environ.get("LLM_API_KEY"), temperature=0)

# --- Инструменты предметной области (магазин) ---
CATALOG = {"P-300": ("USB-C Hub", 2490)}
ORDERS = {"ORD-1002": {"sku": "P-300", "qty": 2, "status": "shipped"}}
POLICIES = {"refund": "Возврат в течение 14 дней после доставки.",
            "shipping": "Стандартная доставка 3–5 дней."}

@tool
def search_products(query: str) -> list:
    """Найти товары по названию."""
    return [{"sku": s, "name": n, "price": p} for s, (n, p) in CATALOG.items() if query.lower() in n.lower()]
@tool
def get_order(order_id: str) -> dict:
    """Информация о заказе."""
    return {"order_id": order_id, **ORDERS.get(order_id, {"error": "not_found"})}
@tool
def get_policy(topic: str) -> str:
    """Политика магазина: refund или shipping."""
    return POLICIES.get(topic, "не найдено")
print("Готово.")

Готово.


## Специалисты с ограниченными инструментами

Каждый специалист — отдельный ReAct-агент, собранный тем же `create_agent`, но **с урезанным списком инструментов**:
- `create_agent(llm, [search_products])` — агент видит только поиск по каталогу;
- `create_agent(llm, [get_order])` — только заказы; и т. д.

Это и есть **специализация**: узкий агент с 1–2 инструментами ошибается реже, чем один агент, которому вывалили сразу все инструменты (ему труднее выбрать правильный). Набор инструментов, переданный в `create_agent`, полностью определяет «компетенцию» агента.

Вспомогательная функция `ask(agent, query)` просто прячет вызов `agent.invoke({"messages": [("user", query)]})` и достаёт из ответа `["messages"][-1].content` — финальный текст. Ниже она используется и напрямую, и внутри вершин графа.

In [2]:
from langchain.agents import create_agent

catalog_agent = create_agent(llm, [search_products])          # каталог
orders_agent  = create_agent(llm, [get_order])                # заказы
policy_agent  = create_agent(llm, [get_policy])               # политики

def ask(agent, query: str) -> str:
    return agent.invoke({"messages": [("user", query)]})["messages"][-1].content

print(ask(catalog_agent, "Есть ли USB-C хаб?")[:120])

Да, USB-C хабы есть в наличии. Если хотите, могу помочь подобрать подходящий вариант. Уточните, пожалуйста, для каких за


## Граф-супервизор: параллельные специалисты + координатор

В примере M2/05 состоянием был готовый `MessagesState`. Здесь состояние своё, и появляются две новые важные идеи LangGraph — **reducer** и **параллельные ветви**. Разберём каждое имя из кода ниже.

**Своя схема состояния — `TypedDict`.**
- `MASState(TypedDict)` описывает, какие поля есть в состоянии графа: `query` (входной запрос), `results` (список ответов специалистов), `answer` (итог координатора). `TypedDict` (из `typing`) — это просто «словарь с объявленными полями и типами»; LangGraph по нему понимает форму состояния.

**Reducer — как объединять обновления поля.**
- `results: Annotated[list, add]` — ключевая конструкция. `Annotated[Тип, reducer]` говорит LangGraph: у поля `results` есть **reducer** — функция, которая *объединяет* старое значение с тем, что вернула вершина. Здесь reducer — `add` (из модуля `operator`, это `+` для списков), поэтому обновления **склеиваются** (конкатенация списков), а не перезаписывают друг друга.
- Зачем это нужно: несколько специалистов работают **одновременно**, и каждый возвращает `{"results": [свой_ответ]}`. Без reducer их обновления затёрли бы друг друга (осталось бы одно). С reducer `add` все ответы аккуратно накапливаются в один список. (Поля без `Annotated`, как `query`/`answer`, обновляются обычным перезаписыванием.)

**Вершины.**
- `specialist_node(agent, name)` — фабрика: возвращает функцию-вершину, которая зовёт своего специалиста через `ask(...)` и дописывает результат в `results`.
- `coordinator(state)` — вершина-агрегатор: склеивает все `results` и одним вызовом `llm.invoke([SystemMessage(...), HumanMessage(...)])` просит модель собрать единый ответ. `llm.invoke` принимает список сообщений; `SystemMessage` — инструкция (роль system), `HumanMessage` — данные (роль user); у ответа берём `.content`.

**Параллельные рёбра (fan-out) и схождение.**
- В цикле для каждого специалиста делаем `g.add_edge(START, name)` — несколько рёбер из одной точки `START`. LangGraph видит, что из входа выходят сразу три ветви, и запускает эти вершины **параллельно** (в один «супершаг»). Их обновления состояния сливаются через reducer.
- `g.add_edge(name, "coordinator")` — все ветви ведут в координатор. LangGraph сам **дожидается завершения всех** входящих ветвей, прежде чем выполнить `coordinator` (схождение). Так получается иерархия: специалисты не общаются между собой, только сходятся к координатору.
- `g.add_edge("coordinator", END)` — после агрегации граф завершается.

**Компиляция и интроспекция.**
- `g.compile()` — собрать исполняемый граф.
- `mas.get_graph().nodes` — служебный метод: возвращает список вершин графа (для проверки/визуализации); в нём видны и служебные `__start__` / `__end__`.

In [3]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage

class MASState(TypedDict):
    query: str
    results: Annotated[list, add]     # ветви параллельно дописывают сюда
    answer: str

def specialist_node(agent, name):
    def node(state: MASState):
        return {"results": [f"[{name}] {ask(agent, state['query'])}"]}
    return node

def coordinator(state: MASState):
    joined = "\n".join(state["results"])
    msg = llm.invoke([SystemMessage("Ты — координатор. Собери из результатов специалистов "
                                    "единый ответ, покрывающий все части запроса."),
                      HumanMessage(f"Запрос: {state['query']}\n\nРезультаты:\n{joined}")])
    return {"answer": msg.content}

g = StateGraph(MASState)
for name, agent in [("catalog", catalog_agent), ("orders", orders_agent), ("policy", policy_agent)]:
    g.add_node(name, specialist_node(agent, name))
    g.add_edge(START, name)               # fan-out: все специалисты стартуют параллельно
    g.add_edge(name, "coordinator")       # сходятся к координатору
g.add_node("coordinator", coordinator)
g.add_edge("coordinator", END)
mas = g.compile()
print("Граф МАС собран. Узлы:", list(mas.get_graph().nodes)[:8])

Граф МАС собран. Узлы: ['__start__', 'catalog', 'orders', 'policy', 'coordinator', '__end__']


## Запуск иерархической МАС

In [4]:
QUERY = ("Хочу узнать статус заказа ORD-1002, есть ли в наличии USB-C хаб и "
         "какие правила возврата.")
out = mas.invoke({"query": QUERY, "results": []})
print("=== Результаты специалистов ===")
for r in out["results"]:
    print(" -", r[:100])
print("\n=== Ответ координатора (иерархическая МАС) ===")
print(out["answer"][:500])

=== Результаты специалистов ===
 - [catalog] Пока я не получил информацию о статусе заказа ORD-1002, так как не знаю, в каком магазине 
 - [orders] Ваш заказ ORD-1002 со статусом "отправлен". В заказе есть 2 USB-C хаба (артикул P-300).

Чт
 - [policy] Пожалуйста, уточните, в каком магазине или у какого продавца был сделан заказ ORD-1002, что

=== Ответ координатора (иерархическая МАС) ===
Ваш заказ ORD-1002 имеет статус «отправлен». В заказе есть 2 USB-C хаба (артикул P-300), которые были в наличии и отправлены вам.

USB-C хабы в целом есть в наличии.

Правила возврата обычно предусматривают возможность вернуть товар в течение 14 дней после доставки при сохранении товарного вида, упаковки и наличии чека. Стандартная доставка занимает 3–5 дней.

Если вы уточните, в каком магазине или сервисе был сделан заказ ORD-1002, мы сможем предоставить более точную информацию по статусу заказ


## Итоги

- Специалисты — независимые агенты (`create_agent`) с **ограниченными** инструментами (специализация).
- **Координатор — это граф** LangGraph: параллельный fan-out к специалистам + узел агрегации; аккумуляция результатов ветвей через reducer состояния.
- Иерархия обеспечивается структурой графа: специалисты сходятся только к координатору, не общаясь напрямую.
- LangGraph берёт на себя оркестрацию (параллелизм, состояние, порядок), оставляя вам проектирование ролей и рёбер. Критик (тема 10) добавляется ещё одним узлом между `coordinator` и `END`.

### Справочник: что нового использовали (сверх примера M2/05)

| Имя | Библиотека | Что делает |
|-----|-----------|-----------|
| `create_agent(llm, [подмножество_инструментов])` | LangChain | специалист: агент с урезанной «компетенцией» |
| `llm.invoke([SystemMessage, HumanMessage])` | LangChain | прямой вызов модели списком сообщений (в координаторе) |
| `class State(TypedDict)` | typing | своя схема состояния графа (поля и типы) |
| `Annotated[list, add]` | typing + LangGraph | поле с **reducer**: обновления *склеиваются*, а не перезаписываются |
| `operator.add` | stdlib | reducer-функция (конкатенация списков) для параллельных ветвей |
| несколько `add_edge(START, name)` | LangGraph | **fan-out**: параллельный запуск ветвей |
| несколько `add_edge(name, "coordinator")` | LangGraph | схождение: координатор ждёт все ветви |
| `graph.get_graph().nodes` | LangGraph | список вершин (интроспекция/визуализация) |

Ключевая мысль: **reducer + параллельные рёбра** — это механизм LangGraph, которым в МАС собирают результаты нескольких агентов, работающих одновременно.